# Word-As-Image — Colab Runner
Make sure **Runtime → Change runtime type → GPU** is enabled before running.

In [5]:
# 1. Check GPU
!nvidia-smi

zsh:1: command not found: nvidia-smi


In [ ]:
# 2. Clone your fork
%cd /content
!git clone https://github.com/jollyland24/Word-As-Image.git
%cd Word-As-Image

In [ ]:
# 3. Install Python dependencies
!pip install -q diffusers==0.21.4 transformers scipy ftfy accelerate
!pip install -q svgwrite svgpathtools cssutils numba torch-tools scikit-fmm easydict visdom freetype-py shapely
!pip install -q opencv-python kornia==0.6.8 wandb huggingface_hub==0.23.2

In [ ]:
# 4. Build diffvg (CUDA-enabled on Colab)
%cd /content/Word-As-Image
!git clone https://github.com/BachiLi/diffvg.git
%cd diffvg
!git submodule update --init --recursive
!python setup.py install
%cd /content/Word-As-Image

In [6]:
# 5. Fix np.bool deprecation in losses.py
import re
with open('code/losses.py', 'r') as f:
    src = f.read()
src = src.replace('dtype=np.bool)', 'dtype=bool)')
src = src.replace('use_auth_token=cfg.token', 'token=cfg.token')
with open('code/losses.py', 'w') as f:
    f.write(src)
print('losses.py patched')

losses.py patched


In [ ]:
# 6. HuggingFace token — paste yours here
HF_TOKEN = ""  # <-- paste your token from https://huggingface.co/settings/tokens

with open('TOKEN', 'w') as f:
    f.write(HF_TOKEN)
print('Token saved')

Token saved


In [9]:
# 7. (Optional) Upload a custom font
import os
from google.colab import files

print('Available fonts:')
for f in os.listdir('code/data/fonts'):
    print(' ', f.replace('.ttf', ''))

print('\nUpload a .ttf file to add a custom font (or skip):')
uploaded = files.upload()
for fname, data in uploaded.items():
    dest = f'code/data/fonts/{fname}'
    with open(dest, 'wb') as out:
        out.write(data)
    print(f'Saved: {dest}')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# 8. Configure generation parameters
import os

SEMANTIC_CONCEPT = "BUNNY"   # concept to visualize
WORD            = "BUNNY"   # word to render (must contain the letter below)
OPTIMIZED_LETTER = "Y"      # single letter to deform
FONT            = "KaushanScript-Regular"  # font name (no .ttf)
SEED            = 0
EXPERIMENT      = "conformal_0.5_dist_pixel_100_kernel201"

# Validate
font_path = f'code/data/fonts/{FONT}.ttf'
assert os.path.exists(font_path), f'Font not found: {font_path}'
assert OPTIMIZED_LETTER in WORD, f'{OPTIMIZED_LETTER} must be in {WORD}'
print('Config OK')

In [ ]:
# 9. Run generation
!python code/main.py \
    --experiment {EXPERIMENT} \
    --semantic_concept {SEMANTIC_CONCEPT} \
    --word {WORD} \
    --optimized_letter {OPTIMIZED_LETTER} \
    --font {FONT} \
    --seed {SEED} \
    --use_wandb 0

In [ ]:
# 10. Display result
import glob
from IPython.display import SVG, Image, display

svg_files = glob.glob(f'output/**/{FONT}_{WORD}_{OPTIMIZED_LETTER}.svg', recursive=True)
png_files = glob.glob(f'output/**/{FONT}_{WORD}_{OPTIMIZED_LETTER}.png', recursive=True)

if svg_files:
    print('SVG output:')
    display(SVG(filename=svg_files[0]))

if png_files:
    print('PNG output:')
    display(Image(filename=png_files[0]))

In [ ]:
# 11. Download results
import shutil
from google.colab import files

zip_name = f'{FONT}_{WORD}_{OPTIMIZED_LETTER}_output.zip'
result_dirs = glob.glob(f'output/**/*concept_{SEMANTIC_CONCEPT}*', recursive=True)
if result_dirs:
    shutil.make_archive(zip_name.replace('.zip',''), 'zip', result_dirs[0])
    files.download(zip_name)
else:
    print('No output directory found yet')